# Time, frequency, and WDM translation

Use one `transform` function for `L1Data` and `BlockState`.
Then let `Wheel` prepare a different representation for each block.

This walkthrough builds synthetic A/E TDI channels, chooses WDM divisions in two
ways, checks round trips, and runs three blocks with local diagnostic
state. The final blocks publish zero signals to demonstrate the exchange
contract; they do not fit a physical source model.

## Setup

Run these commands in a terminal **from the enchilada repository root**. They
assume your WDM checkout is the sibling directory `../wdm`, corresponding to
`~/Documents/lisa/wdm` in your setup.

```sh
uv sync --python 3.13 --extra examples
uv pip install ../wdm matplotlib
uv run --no-sync jupyter lab examples/domain_translation.ipynb
```

Select the kernel from that environment and run the cells from top to bottom.
WDM requires Python 3.13 or newer; Matplotlib is used only for this notebook's
plots. Install the unpublished WDM backend from your checkout. Install it after
project sync and launch with `--no-sync` so the environment retains it.

In [ ]:
import sys
from dataclasses import dataclass, field

import matplotlib.pyplot as plt
import numpy as np

from enchilada import (
    BlockState,
    Covariance,
    L1Data,
    Wheel,
    transform,
)

if sys.version_info < (3, 13):
    raise RuntimeError("Choose a Python >=3.13 kernel for the local WDM backend.")

print(f"Kernel: {sys.executable}")

## 1. Create observations and their covariance

Use 128 samples with a two-second sampling interval. A non-unit interval makes
the normalization visible: enchilada stores Fourier coefficients as
`sample_interval_s * rfft(time_series)`.

Here the two channels have independent Gaussian noise with variance 0.04 per
time sample; `Covariance.from_variance` describes it on the time grid.
Channel order is inferred from the A/E dictionary; observation settings are keyword-only.

In [ ]:
rng = np.random.default_rng(7)
num_time_samples = 128
sample_interval_s = 2.0
sample_times_s = np.arange(num_time_samples) * sample_interval_s
channel_names = ("A", "E")
noise_standard_deviation = 0.2

injected_signal = {
    "A": np.sin(2 * np.pi * 0.035 * sample_times_s),
    "E": 0.6 * np.cos(2 * np.pi * 0.055 * sample_times_s + 0.4),
}
observed = L1Data(
    {
        name: values + rng.normal(0.0, noise_standard_deviation, num_time_samples)
        for name, values in injected_signal.items()
    },
    sample_rate_hz=1.0 / sample_interval_s,
    tdi_generation="2.0",
    physical_observable="strain",
    start_time_gps=100.0,
)
covariance = Covariance.from_variance(
    reference_data=observed,
    time_sample_variance=noise_standard_deviation**2,
)
print(
    f"Domain: {observed.data_domain}; channel A shape: {observed.channel_data['A'].shape}"
)

## 2. Choose either WDM division count

The counts are linked by

```text
num_time_samples = num_frequency_divisions * num_time_divisions
```

Both counts must be positive even integers. Supply either one and the other is
derived. For our 128 samples, eight frequency divisions produce 16 time
divisions; eight time divisions produce 16 frequency divisions.

Each channel's WDM array has shape `(num_frequency_divisions + 1,
num_time_divisions)`. The extra row belongs to the Wilson basis's edge layout.
`L1Data.wdm_grid` is read-only metadata derived automatically from this layout,
and is `None` for time or frequency data. Configure divisions through `transform`
or `Wheel.add(data_domain="wdm", ...)`; do not pass a grid to `L1Data`.
No padding or trimming is performed.

In [ ]:
spectrum = transform(observed, "frequency")
wdm_data = transform(observed, "wdm", num_frequency_divisions=8)
wdm_finer_frequency = transform(observed, "wdm", num_time_divisions=8)

for label, representation in (
    ("Time", observed),
    ("Frequency", spectrum),
    ("WDM: frequency selector", wdm_data),
    ("WDM: time selector", wdm_finer_frequency),
):
    print(f"{label:26s} shape={representation.channel_data['A'].shape}")
    if representation.wdm_grid is not None:
        print(f"  {representation.wdm_grid}")

assert wdm_data.channel_data["A"].shape == (9, 16)
assert wdm_finer_frequency.channel_data["A"].shape == (17, 8)

The two grids trade time resolution for frequency resolution. The WDM plots use
coefficient indices, and all representations show channel A. Inactive WDM edge
entries are masked in the plots; they are structural storage positions, not
missing observations or a statistical noise mask.

In [ ]:
figure, axes = plt.subplots(2, 2, figsize=(12, 7), constrained_layout=True)
axes[0, 0].plot(sample_times_s, observed.channel_data["A"])
axes[0, 0].set(
    title="Observed channel A", xlabel="Time from start [s]", ylabel="Strain"
)

frequencies_hz = np.fft.rfftfreq(num_time_samples, sample_interval_s)
axes[0, 1].plot(frequencies_hz, np.abs(spectrum.channel_data["A"]))
axes[0, 1].set(
    title="Fourier coefficient magnitude", xlabel="Frequency [Hz]", ylabel="|dt · rFFT|"
)

coefficient_limit = max(
    float(np.max(np.abs(representation.channel_data["A"])))
    for representation in (wdm_data, wdm_finer_frequency)
)
for axis, representation in zip(axes[1], (wdm_data, wdm_finer_frequency), strict=True):
    grid = representation.wdm_grid
    visible_coefficients = np.ma.array(
        representation.channel_data["A"], mask=~grid.active_mask
    )
    mesh = axis.imshow(
        visible_coefficients,
        origin="lower",
        aspect="auto",
        interpolation="nearest",
        cmap="RdBu_r",
        vmin=-coefficient_limit,
        vmax=coefficient_limit,
    )
    axis.set(
        title=f"WDM: {grid.num_frequency_divisions} × {grid.num_time_divisions} divisions",
        xlabel="Time division index",
        ylabel="Frequency row index",
    )
    figure.colorbar(mesh, ax=axis, label="WDM coefficient")
plt.show()

## 3. Round trips, regridding, and independent copies

The same wrapper converts back to time or Fourier space. A WDM-to-WDM call can
select a different grid for the same underlying samples. Without new division
counts, it returns an independent copy on the existing grid.

In [ ]:
restored_from_frequency = transform(spectrum, "time")
restored_from_wdm = transform(wdm_data, "time")
regridded_wdm = transform(wdm_data, "wdm", num_time_divisions=8)
copied_wdm = transform(wdm_data, "wdm")

for label, restored in (
    ("Fourier round trip", restored_from_frequency),
    ("WDM round trip", restored_from_wdm),
):
    maximum_error = max(
        float(np.max(np.abs(restored.channel_data[name] - observed.channel_data[name])))
        for name in channel_names
    )
    print(f"{label}: maximum absolute error = {maximum_error:.3e}")
    for name in channel_names:
        np.testing.assert_allclose(
            restored.channel_data[name], observed.channel_data[name], atol=1e-12
        )
    assert restored.start_time_gps == observed.start_time_gps
    assert restored.num_time_samples == observed.num_time_samples

for name in channel_names:
    np.testing.assert_allclose(
        regridded_wdm.channel_data[name],
        wdm_finer_frequency.channel_data[name],
        atol=1e-12,
    )
    assert not np.shares_memory(
        copied_wdm.channel_data[name], wdm_data.channel_data[name]
    )
print("Regridding preserves the samples; copies own their arrays.")

## 4. Covariance translation

Covariance translation between representations is the job of a separate
translation layer; `transform` on a `Covariance` raises `NotImplementedError`
across domains until it is installed, so this walkthrough translates data and
block states only.

`wdm_data.wdm_grid.active_mask` marks the grid's structurally present
coefficients; a WDM `Covariance` must have zero variance exactly where that mask
is `False` (`check_compatible` enforces this).

In [ ]:
covariance.check_compatible(observed)
np.testing.assert_allclose(
    covariance.noise_variance("A", observed), noise_standard_deviation**2
)
print("Covariance on the", observed.data_domain, "grid:", covariance.A.shape)

## 5. Translate a complete BlockState

A `BlockState` carries a `template`, optional `covariance`, and physical
`parameters` (a state carrying a `covariance` cannot cross domains until
covariance translation is available, so this example omits it). Translation
preserves the parameters and transforms the physical arrays. Sampler continuation
and diagnostics belong to the model's internal `State`, which is not passed
through this conversion.

For a state containing template arrays, pass its source `L1Data` as
`reference_data`; `BlockState` itself carries no observation-grid metadata.

In [ ]:
block_state = observed.block_state(
    template=injected_signal,
    parameters={"amplitudes": np.array([1.0, 0.6])},
)
wdm_result = transform(
    block_state,
    "wdm",
    num_frequency_divisions=8,
    reference_data=observed,
)
restored_result = transform(wdm_result, "time", reference_data=wdm_data)

for name in channel_names:
    np.testing.assert_allclose(
        restored_result.template[name], injected_signal[name], atol=1e-12
    )
np.testing.assert_array_equal(
    restored_result.parameters["amplitudes"],
    block_state.parameters["amplitudes"],
)
assert not np.shares_memory(
    wdm_result.parameters["amplitudes"],
    block_state.parameters["amplitudes"],
)
print("WDM signal shape:", wdm_result.template["A"].shape)
print("Restored parameters:", restored_result.parameters)

## 6. Let Wheel translate each block's inputs

The following blocks validate their input representation and publish zero
templates. Each owns a nested `State` with its call count and input diagnostics;
only the physical `BlockState` is stored in the ledger. No block needs to call
`transform` or perform cross-block subtraction.

The caller prepares a zero template on each selected grid and supplies it as
`initial_block_state`. Registration does not run the block. Use a separate block
instance for each independent campaign, because internal state belongs to it.

In [ ]:
@dataclass
class DomainProbe:
    name: str
    expected_domain: str

    @dataclass
    class State:
        num_sample_calls: int = 0
        input_domain: str | None = None
        input_shape: tuple[int, ...] | None = None

    state: State = field(default_factory=State, init=False)

    def sample(
        self, conditional_residual, covariance, current_block_state, *, rng
    ) -> BlockState:
        assert conditional_residual.data_domain == self.expected_domain
        # Covariance translation across domains arrives with the translation
        # layer (design spec section 6); this walkthrough runs without one.
        assert covariance is None
        for name in conditional_residual.channel_names:
            assert (
                current_block_state.template[name].shape
                == conditional_residual.channel_data[name].shape
            )
        result = conditional_residual.zero_block_state()
        self.state = self.State(
            self.state.num_sample_calls + 1,
            conditional_residual.data_domain,
            conditional_residual.channel_data["A"].shape,
        )
        return result

In [ ]:
wheel = Wheel(observed, random_seed=3)
blocks = {}
for reference_data in (observed, spectrum, wdm_finer_frequency):
    domain = reference_data.data_domain
    initial = reference_data.zero_block_state()
    block = DomainProbe(f"{domain}_block", domain)
    blocks[block.name] = block
    options = {"num_time_divisions": 8} if domain == "wdm" else {}
    wheel.add(
        block,
        initial_block_state=initial,
        data_domain=domain,
        **options,
    )
wheel.run(num_cycles=2)

for name, result in wheel.ledger.snapshot().items():
    state = blocks[name].state
    print(
        f"{name:16s} receives {state.input_domain:9s} "
        f"{str(state.input_shape):10s}; "
        f"ledger signal shape {result.template['A'].shape}; "
        f"sample calls {state.num_sample_calls}"
    )
    assert state.num_sample_calls == 2
    assert result.template["A"].shape == observed.channel_data["A"].shape

assert wheel.working_residual.data_domain == "time"
for name in channel_names:
    np.testing.assert_array_equal(
        wheel.working_residual.channel_data[name], observed.channel_data[name]
    )
print("Canonical domain:", wheel.working_residual.data_domain)
print("The zero-signal blocks leave the observations and full residual unchanged.")

Wheel stores accepted estimates on the original observation grid. Before each
call, it translates the conditional residual and the block's previous signal
into the registered representation. Returned estimates are converted back
before adoption. Parameters and sampler state retain their block-local meaning.

This campaign has no covariance, because translating one across domains needs
the covariance translation layer. A block that owns the noise model must return
its complete covariance on every call. Omitting `data_domain` in `Wheel.add`
preserves the existing input representations rather than selecting a new one.

See the [README translation section](../README.md#time-frequency-and-wdm-translation) for conventions
and covariance semantics, or [the Python script](domain_translation.py) for a
compact command-line walkthrough.